### ***Gold DimDate — Date Dimension***

In [4]:
from pyspark.sql.functions import (
    col,
    min as spark_min,
    max as spark_max,
    explode,
    sequence,
    to_date,
    year,
    quarter,
    month,
    dayofmonth,
    dayofyear,
    dayofweek,
    weekofyear,
    date_format,
    concat,
    lit
)
from datetime import date

# ============================================================
# 1. CONFIGURATION
# ============================================================

catalog_db = "FABRIC_COE_WORKLOAD.stock_market_analysis_LH"

gold_performance = (
    f"{catalog_db}.gold.gld_daily_stock_performance"
)

gold_sentiment = (
    f"{catalog_db}.gold.gld_stock_sentiment_analysis"
)

gold_forecast = (
    f"{catalog_db}.gold.gld_stock_price_forecast"
)

dim_date_table = f"{catalog_db}.gold.gld_dim_date"

print("Starting Gold DimDate processing...")

# ============================================================
# 2. READ DATES FROM ALL THREE GOLD TABLES
# ============================================================

performance_dates = (
    spark.table(gold_performance)
    .select(to_date(col("trade_date")).alias("date"))
)

sentiment_dates = (
    spark.table(gold_sentiment)
    .select(to_date(col("trade_date")).alias("date"))
)

forecast_dates = (
    spark.table(gold_forecast)
    .select(to_date(col("forecast_date")).alias("date"))
)

# Combine all dates
all_gold_dates = (
    performance_dates
    .unionByName(sentiment_dates)
    .unionByName(forecast_dates)
    .filter(col("date").isNotNull())
)

# ============================================================
# 3. FIND MINIMUM AND MAXIMUM DATES
# ============================================================

date_range = (
    all_gold_dates
    .agg(
        spark_min("date").alias("min_date"),
        spark_max("date").alias("max_date")
    )
    .first()
)

min_date = date_range["min_date"]
max_date = date_range["max_date"]

if min_date is None or max_date is None:
    raise ValueError(
        "No valid dates found in the Gold tables. "
        "Ensure the Gold tables contain valid dates."
    )

# Extend the range to complete calendar years
start_date = date(min_date.year, 1, 1)
end_date = date(max_date.year, 12, 31)

print(f"Earliest Gold date: {min_date}")
print(f"Latest Gold date:   {max_date}")
print(f"Calendar start:     {start_date}")
print(f"Calendar end:       {end_date}")

# ============================================================
# 4. GENERATE A CONTINUOUS CALENDAR
# ============================================================

days_df = spark.sql(f"""
    SELECT EXPLODE(
        SEQUENCE(
            TO_DATE('{start_date}'),
            TO_DATE('{end_date}'),
            INTERVAL 1 DAY
        )
    ) AS date
""")

# ============================================================
# 5. CREATE DATE DIMENSION ATTRIBUTES
# ============================================================

dim_date_df = (
    days_df
    .select(
        # Date and numeric key
        col("date"),
        date_format("date", "yyyyMMdd")
            .cast("int")
            .alias("DateKey"),

        # Year and quarter
        year("date").alias("Year"),
        quarter("date").alias("QuarterNumber"),
        concat(
            year("date").cast("string"),
            lit("-Q"),
            quarter("date").cast("string")
        ).alias("YearQuarter"),

        # Month
        month("date").alias("MonthNumber"),
        date_format("date", "MMMM").alias("MonthName"),
        date_format("date", "MMM").alias("MonthShort"),
        date_format("date", "yyyy-MM").alias("YearMonth"),

        # Day
        dayofmonth("date").alias("Day"),
        dayofyear("date").alias("DayOfYear"),

        # Day of week: Sunday = 1, Saturday = 7
        dayofweek("date").alias("DayOfWeekNumber"),
        date_format("date", "EEEE").alias("DayOfWeek"),
        date_format("date", "EEE").alias("DayOfWeekShort"),

        # Week of year
        weekofyear("date").alias("WeekOfYear")
    )
)

# ============================================================
# 6. VALIDATE THE GENERATED DATE DIMENSION
# ============================================================

if dim_date_df.filter(
    col("date").isNull() | col("DateKey").isNull()
).limit(1).count() > 0:
    raise ValueError("DimDate contains null dates or DateKeys.")

duplicate_keys = (
    dim_date_df
    .groupBy("DateKey")
    .count()
    .filter(col("count") > 1)
)

if duplicate_keys.limit(1).count() > 0:
    raise ValueError("Duplicate DateKeys found in generated DimDate.")

dim_date_df.createOrReplaceTempView("vw_dim_date_source")

print("Date dimension generated and validated successfully.")

# ============================================================
# 7. INITIAL LOAD OR SCD TYPE 1 MERGE
# ============================================================

if not spark.catalog.tableExists(dim_date_table):

    # Initial load
    (
        dim_date_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(dim_date_table)
    )

    print(
        f"Created {dim_date_table} successfully."
    )

else:

    print("DimDate exists. Applying SCD Type 1 MERGE.")

    # Check that the existing table has the expected schema
    required_columns = set(dim_date_df.columns)
    existing_columns = set(
        spark.table(dim_date_table).columns
    )

    missing_columns = required_columns - existing_columns

    if missing_columns:
        raise ValueError(
            "Existing DimDate table is missing columns: "
            + ", ".join(sorted(missing_columns))
            + ". Migrate the schema before running the MERGE."
        )

    # Ensure the existing dimension key is unique
    duplicate_target_keys = (
        spark.table(dim_date_table)
        .groupBy("DateKey")
        .count()
        .filter(col("count") > 1)
    )

    if duplicate_target_keys.limit(1).count() > 0:
        raise ValueError(
            "Existing DimDate contains duplicate DateKeys. "
            "Resolve duplicates before running MERGE."
        )

    # Update existing dates and insert newly required dates
    spark.sql(f"""
        MERGE INTO {dim_date_table} AS target
        USING vw_dim_date_source AS source
        ON target.DateKey = source.DateKey

        WHEN MATCHED THEN UPDATE SET
            target.date = source.date,
            target.Year = source.Year,
            target.QuarterNumber = source.QuarterNumber,
            target.YearQuarter = source.YearQuarter,
            target.MonthNumber = source.MonthNumber,
            target.MonthName = source.MonthName,
            target.MonthShort = source.MonthShort,
            target.YearMonth = source.YearMonth,
            target.Day = source.Day,
            target.DayOfYear = source.DayOfYear,
            target.DayOfWeekNumber = source.DayOfWeekNumber,
            target.DayOfWeek = source.DayOfWeek,
            target.DayOfWeekShort = source.DayOfWeekShort,
            target.WeekOfYear = source.WeekOfYear

        WHEN NOT MATCHED THEN INSERT (
            date,
            DateKey,
            Year,
            QuarterNumber,
            YearQuarter,
            MonthNumber,
            MonthName,
            MonthShort,
            YearMonth,
            Day,
            DayOfYear,
            DayOfWeekNumber,
            DayOfWeek,
            DayOfWeekShort,
            WeekOfYear
        )
        VALUES (
            source.date,
            source.DateKey,
            source.Year,
            source.QuarterNumber,
            source.YearQuarter,
            source.MonthNumber,
            source.MonthName,
            source.MonthShort,
            source.YearMonth,
            source.Day,
            source.DayOfYear,
            source.DayOfWeekNumber,
            source.DayOfWeek,
            source.DayOfWeekShort,
            source.WeekOfYear
        )
    """)

    print("DimDate SCD Type 1 MERGE completed successfully.")

# ============================================================
# 8. FINAL VALIDATION
# ============================================================

final_dim_date = spark.table(dim_date_table)

total_dates = final_dim_date.count()
distinct_dates = final_dim_date.select("date").distinct().count()
distinct_keys = final_dim_date.select("DateKey").distinct().count()

print(f"DimDate table: {dim_date_table}")
print(f"Total rows: {total_dates}")
print(f"Distinct dates: {distinct_dates}")
print(f"Distinct DateKeys: {distinct_keys}")

if total_dates != distinct_dates or total_dates != distinct_keys:
    raise ValueError(
        "DimDate validation failed: duplicate dates or DateKeys exist."
    )

display(
    final_dim_date
    .orderBy("date")
)


StatementMeta(, 22449ee8-6e67-46b1-94eb-04d2da5cb7fc, 6, Finished, Available, Finished, True)

Starting Gold DimDate processing...
Earliest Gold date: 2021-10-25
Latest Gold date:   2026-11-03
Calendar start:     2021-01-01
Calendar end:       2026-12-31
Date dimension generated and validated successfully.
DimDate exists. Applying SCD Type 1 MERGE.
DimDate SCD Type 1 MERGE completed successfully.
DimDate table: FABRIC_COE_WORKLOAD.stock_market_analysis_LH.gold.gld_dim_date
Total rows: 2191
Distinct dates: 2191
Distinct DateKeys: 2191


SynapseWidget(Synapse.DataFrame, 36612799-2fe0-4b9f-8d5f-da2cadc86a7e)

### ***Total rows in Silver table***

In [2]:
table_name = "FABRIC_COE_WORKLOAD.stock_market_analysis_LH.silver.slv_stock_prices"

row_count = spark.table(table_name).count()

print(f"Table: {table_name}")
print(f"Row Count: {row_count:,}")

StatementMeta(, 5fdf3887-c87e-4c73-bd40-f09ad657cf49, 4, Finished, Available, Finished, False)

Table: FABRIC_COE_WORKLOAD.stock_market_analysis_LH.silver.slv_stock_prices
Row Count: 30,193
